In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# ── imports ──────────────────────────────────────────────
import asyncio
from pylabrobot.resources.coordinate import Coordinate
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import (
    STARLetDeck, PLT_CAR_L5MD_A00, TIP_CAR_480_A00               
)

from pylabrobot.resources import hamilton_96_tiprack_50uL_filter

# ── build LH + deck ──────────────────────────────────────
backend = STARBackend()
lh      = LiquidHandler(backend=backend, deck=STARLetDeck())

await lh.setup(skip_autoload=True)   

2026-02-17 09:01:38,243 - pylabrobot.io.usb - INFO - Finding USB device...
2026-02-17 09:01:38,251 - pylabrobot.io.usb - INFO - Found USB device.
2026-02-17 09:01:38,254 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-02-17 09:01:41,425 - pylabrobot - INFO - Running backend initialization procedure.


In [3]:
 
from pylabrobot.resources.hamilton import (
  STARLetDeck,
  PLT_CAR_L5MD_A00,
  TIP_CAR_480_A00,
)
from pylabrobot.resources.vwr.plates import VWR_96_wellplate_100uL_Vb
# VWR_96_wellplate_100uL_Vb
# VWR_96_wellplate_100ul_Vb
from pylabrobot.resources.agilent.plates import agilent_96_wellplate_150uL_Vb
from pylabrobot.resources.eppendorf.plates import Eppendorf_96_wellplate_250ul_Vb
from pylabrobot.resources import TIP_50ul_w_filter

# plate adapter import (per your path)
from pylabrobot.resources.hamilton.plate_adapters import Hamilton_96_adapter_188182


# --- carriers ---
plate_car = PLT_CAR_L5MD_A00(name="plate_carrier")
tip_car = TIP_CAR_480_A00(name="tip_carrier")

# place carriers on deck rails
lh.deck.assign_child_resource(plate_car, rails=19)
lh.deck.assign_child_resource(tip_car, rails=25)


# --- resources that sit on carriers ---
plate_adapterVWR = Hamilton_96_adapter_188182(name="plate_adapterVWR")
plateVWR   = VWR_96_wellplate_100uL_Vb(name="VWR_96_wellplate")
# dx_mm=12.5, dy_mm=10,
dx=11.5
dy=9.65
dz=8
# plate_car[0].assign_child_resource(plate_adapterVWR, location=Coordinate(dx, dy, dz))
# plate_car[0].assign_child_resource(plate_adapterVWR)
plate_adapterVWR.assign_child_resource(plateVWR)

# plate_adapter_Epp = Hamilton_96_adapter_188182(name="plate_adapter_Epp")
# plateEpp   = Eppendorf_96_wellplate_250ul_Vb(name="eppendorf_96_plate")
# plate_car[1].assign_child_resource(plate_adapter_Epp)
# plate_adapter_Epp.assign_child_resource(plateEpp)

# plate_adapter = Hamilton_96_adapter_188182(name="plate_adapter")
# plate   = agilent_96_wellplate_150uL_Vb(name="agilent_96_plate")
# plate_car[2].assign_child_resource(plate_adapter)
# plate_adapter.assign_child_resource(plate)



# tips
tips_50 = hamilton_96_tiprack_50uL_filter(name="tips_50ul_filter")
# put tip rack at position[0] on tip carrier 
tip_car[1].assign_child_resource(tips_50)
                




In [4]:
from pylabrobot.resources.coordinate import Coordinate

async def test_5p_geomtetry():
    
    # pick up a single tip using channel 0 from A1 of the tip rack
    await lh.pick_up_tips(
        tips_50["A1"],
        use_channels=[1],
    )

       # move channel 0 to well A1 on the plate adapter
    await lh.dispense(
        plateVWR["A1"],
        vols=[0],
        use_channels=[1],
        # optional: keep it conservative for a first geometry test
        liquid_height=[10],      # mm above well bottom (adjust as you like)
        blow_out=[0],
        settling_time=[5],
    )
    # move channel 0 to well A1 on the plate adapter
    await lh.dispense(
        plateVWR["H1"],
        vols=[0],
        use_channels=[1],
        # optional: keep it conservative for a first geometry test
        liquid_height=[10],      # mm above well bottom (adjust as you like)
        blow_out=[0],
        settling_time=[5],
    )
    await lh.dispense(
        plateVWR["A12"],
        vols=[0],
        use_channels=[1],
        # optional: keep it conservative for a first geometry test
        liquid_height=[10],      # mm above well bottom (adjust as you like)
        blow_out=[0],
        settling_time=[5],
    )
    await lh.dispense(
        plateVWR["H12"],
        vols=[0],
        use_channels=[1],
        # optional: keep it conservative for a first geometry test
        liquid_height=[10],      # mm above well bottom (adjust as you like)
        blow_out=[0],
        settling_time=[5],
    )


In [11]:
async def move_channel_to_well_center(
    lh,
    well,
    channel: int = 1,
    z_clearance: float = 20,
    dx_mm: float = 0.0,
    dy_mm: float = 0.0,
    step_mm: float | None = None,   # e.g. 2.0 to walk in 2mm increments
    descend: bool = False,          # keep False while probing geometry
):
    """
    Move a single channel to a well's computed absolute center, with optional XY offsets.

    dx_mm, dy_mm:
      +dx moves right, +dy moves "forward" (Hamilton Y+) in deck coordinates.
    step_mm:
      if set, moves in small increments to help you visually confirm geometry.
    descend:
      if True, move down to the well's Z (use carefully).
    """
    # Resolve the resource -> absolute deck coordinate
    loc = well.get_absolute_location()
    x0, y0, z0 = loc.x, loc.y, loc.z

    x = x0 + dx_mm
    y = y0 + dy_mm
    z = z0 + 8 # plate offset specific for VWR plates; adjust as needed for specific plate/carrier

    print(f"Well center (computed): x={x0:.2f} y={y0:.2f} z={z0:.2f}")
    print(f"Target (with offsets):  x={x:.2f} y={y:.2f} z={z:.2f}  (dx={dx_mm:+.2f}, dy={dy_mm:+.2f})")

    # Put robot in manual move mode
    await lh.prepare_for_manual_channel_operation(channel)

    # Always go up first
    await lh.move_channel_z(channel, z + z_clearance)

    # Helper to optionally "walk" to the target
    async def _move_axis_in_steps(move_fn, start_val: float, end_val: float, axis_name: str):
        if step_mm is None or step_mm <= 0:
            await move_fn(channel, end_val)
            return

        import math
        delta = end_val - start_val
        if abs(delta) < 1e-6:
            return

        n = int(math.ceil(abs(delta) / step_mm))
        for i in range(1, n + 1):
            v = start_val + (delta * i / n)
            await move_fn(channel, v)
            print(f"  {axis_name} -> {v:.2f}")

    # We don't reliably know current X/Y without a request_* call;
    # for "stepping", we step from the computed well center (x0,y0).
    # If you want true stepping from current position, tell me your backend has request_x/y helpers.
    await _move_axis_in_steps(lh.move_channel_x, x0, x, "X")
    await _move_axis_in_steps(lh.move_channel_y, y0, y, "Y")

    if descend:
        await lh.move_channel_z(channel, z)

    print(f"Moved channel {channel} to x={x:.2f} y={y:.2f} (Z clearance={z_clearance:.2f})")


In [ ]:

# await test_5p_geomtetry()

await lh.pick_up_tips(
    tips_50["A1"],
    use_channels=[1],
)

well = plateVWR["H1"]
if isinstance(well, list):
    well = well[0]

# await move_channel_to_well_center(lh, well, channel=2, z_clearance=20)
await move_channel_to_well_center(lh, well, channel=1, dx_mm=11.5, dy_mm=9.65, descend=True)   #H12, #A12, A1, H1




Well center (computed): x=2.00 y=3.20 z=11.30
Target (with offsets):  x=13.50 y=12.85 z=15.30  (dx=+11.50, dy=+9.65)


STARFirmwareError: {'Pipetting channel 3': CommandSyntaxError('Parameter out of range')}, C0KZid0014er99/00 P301/32

In [10]:
# await lh.discard_tips()
await lh.drop_tips(tips_50["A1"], use_channels=[2])

# # ── park channel 0 on the pedestal centre ───────────────
# await lh.backend.prepare_for_manual_channel_operation(0)

# # centre of the module in deck coordinates
# mod_origin = dwp_mod.get_absolute_location()
# center_x   = mod_origin.x + dwp_mod.get_absolute_size_x() / 2
# center_y   = mod_origin.y + dwp_mod.get_absolute_size_y() / 2
# center_z   = mod_origin.z + dwp_mod.get_absolute_size_z()   

# print (center_x, center_y, center_z)    

# # ── pick up tip from position 2, well A1 ─────────────────
    # tip_spot = tip_rack["A1"][0]                                      # choose the exact well you want
    # await lh.pick_up_tips([tip_spot], use_channels=[0])

    # # ── manual control on channel 0 ──────────────────────────
    # await lh.backend.prepare_for_manual_channel_operation(0)

    # # centre of the module in deck coordinates
    # mod_origin = dwp_mod.get_absolute_location()
    # center_x   = mod_origin.x + dwp_mod.get_absolute_size_x() / 2
    # center_y   = mod_origin.y + dwp_mod.get_absolute_size_y() / 2
    # center_z   = mod_origin.z + dwp_mod.get_absolute_size_z()

    # # 1. traverse in the XY plane
    # await lh.backend.move_channel_x(0, center_x)
    # await lh.backend.move_channel_y(0, center_y)

    # # 2. descend to the pedestal surface
    # await lh.backend.move_channel_z(0, center_z)

    # print ("X coord:", center_x, "Y coord:", center_y, "Z coord:", center_z)



# ── module → carrier → deck ─────────────────────────────
# dwp_mod   = MFX_DWP_module_188042("dwp_mod_1")
# other_mod = MFX_DWP_rackbased_module("dwp_mod_2")
# flex_car  = MFX_CAR_L5_base("flex_car_1", modules={0: dwp_mod, 1: other_mod})
# lh.deck.assign_child_resource(flex_car, rails=13)               # ← rails for the module carrier

# # ── tip carrier with 50 µL tips ─────────────────────────
# tip_car = TIP_CAR_480_A00("tip_car")
# tip_car[1] = TIP_50ul_w_filter(name="tips_01")
# # tip_rack = fifty_ul_tip_with_filter(name="tips_50ul")                       # slot 2 (index 2) on the carrier
# # holder.assign_child_resource(tip_rack)            # mount rack into that holder
# lh.deck.assign_child_resource(tip_car, rails=1)   # place the whole carrier on the deck
# # lh.deck.assign_child_resource(tip_car, rails=1)                 # ← rails for the tip carrier

# # ── initialise hardware (run Autoload!) ─────────────────
# await lh.setup(skip_autoload=True)

# # ── pick up tip from position 2, well A1 ─────────────────
# tiprack = lh.deck.get_resource("tips_01")
# await lh.pick_up_tips(tiprack["A1"], use_channels=[3])

# # # ── manual control on channel 0 ──────────────────────────
# await lh.backend.prepare_for_manual_channel_operation(0)

# # # centre of the module in deck coordinates
# mod_origin = dwp_mod.get_absolute_location()
# center_x   = mod_origin.x + dwp_mod.get_absolute_size_x() / 2
# center_y   = mod_origin.y + dwp_mod.get_absolute_size_y() / 2
# center_z   = mod_origin.z + dwp_mod.get_absolute_size_z()

# print(center_x, center_y, center_z)

# # 1. traverse in the XY plane
# await lh.backend.move_channel_x(0, center_x)# # ── park channel 0 on the pedestal centre ───────────────
# await lh.backend.prepare_for_manual_channel_operation(0)

# # centre of the module in deck coordinates
# mod_origin = dwp_mod.get_absolute_location()
# center_x   = mod_origin.x + dwp_mod.get_absolute_size_x() / 2
# center_y   = mod_origin.y + dwp_mod.get_absolute_size_y() / 2
# center_z   = mod_origin.z + dwp_mod.get_absolute_size_z()   

# print (center_x, center_y, center_z)
# await lh.backend.move_channel_y(0, center_y)

# # 2. descend to the pedestal surface
# await lh.backend.move_channel_z(0, center_z)

# print ("X coord:", center_x, "Y coord:", center_y, "Z coord:", center_z)

# # 3. pause for visual inspection
# await asyncio.sleep(10)

# # 4. lift clear of the module
# await lh.backend.move_channel_z(0, center_z + 30)               # raise 30 mm (adjust if needed)

# # 5. move back above the original tip position
# tip_loc = tip_pos.get_absolute_location()
# await lh.backend.move_channel_x(0, tip_loc.x)
# await lh.backend.move_channel_y(0, tip_loc.y)
# await lh.backend.move_channel_z(0, tip_loc.z + 10)              # hover 10 mm above rack

# # 6. return tip to rack
# await lh.drop_tips([tip_pos], use_channels=[0])


In [ ]:
# await lh.stop()